In [46]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from typing import List, Optional, Union, Dict, Tuple
from tqdm.autonotebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# HOUSTON

## XGBoost

In [152]:
def short_term_forecast_XGB(hour=None):
    # 模型配置（当前最佳）
    model = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        colsample_bylevel=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=10,
        gamma=0,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        # enable_categorical=True,
        random_state=42,
        n_joba=-1,
        verbosity=0,
    )

    X = pd.read_csv('./train_data/dam_features.csv')
    if hour:
        X = X[X.hour==hour]
        _ = X.pop('hour')
    X = X[X.timestamp >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.timestamp < '2025-01-01 00:00:00']
    X_test = X[X.timestamp >= '2025-01-01 00:00:00']
    _ = X_train.pop('timestamp')
    dt = X_test.pop('timestamp')
    y_train = X_train.pop('target')
    y_test = X_test.pop('target')

    if hour:
        cat_features = [0, 1, 2, 3, 5, 6, 7, 8, 31, 32]
    else:
        cat_features = [0, 1, 2, 3, 4, 6, 7, 8, 9, 32, 33]

    # for i in cat_features:
    #     X_train.iloc[:, i] = X_train.iloc[:, i].astype("category")
    #     X_test.iloc[:, i] = X_test.iloc[:, i].astype("category")

    model.fit(
        X_train, y_train
    )
    
    return  model, X_test, y_test, dt

### 01:00 - 24:00

In [153]:
preds = []
y_tests = []
for h in tqdm(range(1, 25)):
    print("************************")
    print(f'开始第{h}小时预测')
    print("************************")
    model1, X_test, y_test, dt = short_term_forecast_XGB(hour=h)

    y_pred = model1.predict(X_test)
    xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
    xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
    native_mae1 = mean_absolute_error(y_test, X_test['dam_lag_24h'])
    native_rmse1 = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
    print("xgb_model_mae1:", round(xgb_model_mae1, 2))
    print("native_mae:", round(native_mae1, 2))

    print("xgb_model_rmse1:", round(xgb_model_rmse1
                                    , 2))
    print("native_rmse:", round(native_rmse1, 2))

    xgb_mae_enhance1 = round((native_mae1 - xgb_model_mae1) / native_mae1 * 100, 2)
    xgb_rmse_enhance1 = round((native_rmse1 - xgb_model_rmse1) / native_rmse1 * 100, 2)
    print('mae 提升:', xgb_mae_enhance1, '%')
    print('rmse 提升:', xgb_rmse_enhance1, '%')
    preds.extend(list(y_pred))
    y_tests.extend(list(y_test))
    print()

  0%|          | 0/24 [00:00<?, ?it/s]

************************
开始第1小时预测
************************
(1431, 36)
xgb_model_mae1: 5.07
native_mae: 7.01
xgb_model_rmse1: 7.82
native_rmse: 10.53
mae 提升: 27.73 %
rmse 提升: 25.7 %

************************
开始第2小时预测
************************
(1431, 36)
xgb_model_mae1: 4.82
native_mae: 6.12
xgb_model_rmse1: 7.25
native_rmse: 9.21
mae 提升: 21.18 %
rmse 提升: 21.24 %

************************
开始第3小时预测
************************
(1427, 36)
xgb_model_mae1: 5.01
native_mae: 5.96
xgb_model_rmse1: 7.59
native_rmse: 9.16
mae 提升: 15.82 %
rmse 提升: 17.16 %

************************
开始第4小时预测
************************
(1431, 36)
xgb_model_mae1: 5.07
native_mae: 5.97
xgb_model_rmse1: 7.68
native_rmse: 9.13
mae 提升: 15.13 %
rmse 提升: 15.87 %

************************
开始第5小时预测
************************
(1431, 36)
xgb_model_mae1: 5.03
native_mae: 6.39
xgb_model_rmse1: 8.11
native_rmse: 9.89
mae 提升: 21.23 %
rmse 提升: 17.99 %

************************
开始第6小时预测
************************
(1431, 36)
xgb_model_mae1: 6.27

In [17]:
y_pred = preds
y_test = y_tests
xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
print("xgb_model_mae1:", xgb_model_mae1)

print("xgb_model_rmse1:", xgb_model_rmse1)

xgb_model_mae1: 8.800295213523574
xgb_model_rmse1: 21.17635063357332


In [154]:
model1, X_test, y_test, dt = short_term_forecast_XGB()

y_pred = model1.predict(X_test)
xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae1 = mean_absolute_error(y_test, X_test['dam_lag_24h'])
native_rmse1 = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
print("xgb_model_mae1:", xgb_model_mae1)
print("native_mae:", native_mae1)

print("xgb_model_rmse1:", xgb_model_rmse1)
print("native_rmse:", native_rmse1)

xgb_mae_enhance1 = round((native_mae1 - xgb_model_mae1) / native_mae1 * 100, 2)
xgb_rmse_enhance1 = round((native_rmse1 - xgb_model_rmse1) / native_rmse1 * 100, 2)
print('mae 提升:', xgb_mae_enhance1, '%')
print('rmse 提升:', xgb_rmse_enhance1, '%')

(34340, 37)
xgb_model_mae1: 8.604912286374715
native_mae: 10.578265705206551
xgb_model_rmse1: 20.853778189985533
native_rmse: 26.220882300682653
mae 提升: 18.65 %
rmse 提升: 20.47 %


In [155]:
xgb_pre = model1.predict(X_test)

### test

In [156]:
X = pd.read_csv('./train_data/dam_features.csv')
X_test = X[X.timestamp >= '2025-01-01 00:00:00']
X_test = X_test.dropna()

In [157]:
y_pred = pd.Series(y_pred)
y_test.reset_index(drop=True, inplace=True)
dt = X_test[['timestamp', 'hour']]
dt.reset_index(drop=True, inplace=True)
result = pd.concat([dt, y_test, y_pred], axis=1)
result.columns = ['dt', 'hour', 'target', 'pred']

In [158]:
result.to_csv("DAM_result.csv", index=None)

In [159]:
result

,dt,hour,target,pred
0,2025-01-01 06:00:00,1,20.14,19.958693
1,2025-01-01 06:00:00,2,17.90,19.083134
2,2025-01-01 06:00:00,3,16.69,18.180393
3,2025-01-01 06:00:00,4,17.02,17.618713
4,2025-01-01 06:00:00,5,18.21,18.464663
...,...,...,...,...
8177,2025-12-09 06:00:00,20,60.05,37.721817
8178,2025-12-09 06:00:00,21,63.34,36.119938
8179,2025-12-09 06:00:00,22,57.15,33.842979
8180,2025-12-09 06:00:00,23,49.72,29.196398


## LightGBM

In [197]:
def short_term_forecast_LGBM(hour=None):
    # 模型配置（当前最佳）
    model = LGBMRegressor(
        n_estimators=1000,
        num_leaves=31,
        max_depth=8,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=20,
        min_child_weight=1e-3,
        objective='mae',
        metric='mae',
        device='gpu',
        boosting_type='gbdt',
        random_state=42,
        n_jobs=4,
        verbose=-1,
        force_row_wise=True,
    )

    X = pd.read_csv('./train_data/dam_features.csv')
    if hour:
        X = X[X.hour==hour]
        _ = X.pop('hour')
    X = X[X.timestamp >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.timestamp < '2025-01-01 00:00:00']
    X_test = X[X.timestamp >= '2025-01-01 00:00:00']
    _ = X_train.pop('timestamp')
    dt = X_test.pop('timestamp')
    y_train = X_train.pop('target')
    y_test = X_test.pop('target')

    if hour:
        cat_features = [0, 1, 2, 3, 5, 6, 7, 8, 31, 32]
    else:
        cat_features = [0, 1, 2, 3, 4, 6, 7, 8, 9, 32, 33]

    # for i in cat_features:
    #     X_train.iloc[:, i] = X_train.iloc[:, i].astype("category")
    #     X_test.iloc[:, i] = X_test.iloc[:, i].astype("category")

    model.fit(
        X_train, y_train,
        # categorical_feature=cat_features
    )
    
    return  model, X_test, y_test, dt

### 01:00 - 24:00

In [161]:
preds = []
y_tests = []
for h in tqdm(range(1, 25)):
    print("************************")
    print(f'开始第{h}小时预测')
    print("************************")
    model1, X_test, y_test, dt = short_term_forecast_LGBM(hour=h)

    y_pred = model1.predict(X_test)
    lgb_model_mae1 = mean_absolute_error(y_test, y_pred)
    lgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
    native_mae = mean_absolute_error(y_test, X_test['dam_lag_24h'])
    native_rmse = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
    print("lgb_model_mae1:", lgb_model_mae1)
    print("native_mae:", native_mae)

    print("lgb_model_rmse1:", lgb_model_rmse1)
    print("native_rmse:", native_rmse)

    lgb_mae_enhance1 = round((native_mae - lgb_model_mae1) / native_mae * 100, 2)
    lgb_rmse_enhance1 = round((native_rmse - lgb_model_rmse1) / native_rmse * 100, 2)
    print('mae 提升:', lgb_mae_enhance1, '%')
    print('rmse 提升:', lgb_rmse_enhance1, '%')
    preds.extend(list(y_pred))
    y_tests.extend(list(y_test))
    print()

  0%|          | 0/24 [00:00<?, ?it/s]

************************
开始第1小时预测
************************
(1431, 36)
lgb_model_mae1: 5.205485948764017
native_mae: 7.0146627565982405
lgb_model_rmse1: 8.180345974777522
native_rmse: 10.530246395481928
mae 提升: 25.79 %
rmse 提升: 22.32 %

************************
开始第2小时预测
************************
(1431, 36)
lgb_model_mae1: 5.068310956320296
native_mae: 6.117096774193548
lgb_model_rmse1: 7.66545171317349
native_rmse: 9.208141415532548
mae 提升: 17.15 %
rmse 提升: 16.75 %

************************
开始第3小时预测
************************
(1427, 36)
lgb_model_mae1: 5.347034396758856
native_mae: 5.956755162241889
lgb_model_rmse1: 8.07457067632227
native_rmse: 9.160040930722273
mae 提升: 10.24 %
rmse 提升: 11.85 %

************************
开始第4小时预测
************************
(1431, 36)
lgb_model_mae1: 5.275584020205186
native_mae: 5.972815249266862
lgb_model_rmse1: 8.020753728330835
native_rmse: 9.126912011951266
mae 提升: 11.67 %
rmse 提升: 12.12 %

************************
开始第5小时预测
************************
(1431

In [162]:
y_pred = preds
y_test = y_tests
xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
print("lgb_model_mae1:", xgb_model_mae1)

print("lgb_model_rmse1:", xgb_model_rmse1)

lgb_model_mae1: 8.75817425999477
lgb_model_rmse1: 21.43795010916479


In [203]:
model1, X_test, y_test, dt = short_term_forecast_LGBM()

y_pred = model1.predict(X_test)
lgb_model_mae1 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test['dam_lag_24h'])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
print("lgb_model_mae1:", lgb_model_mae1)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse1)
print("native_rmse:", native_rmse)

lgb_mae_enhance1 = round((native_mae - lgb_model_mae1) / native_mae * 100, 2)
lgb_rmse_enhance1 = round((native_rmse - lgb_model_rmse1) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance1, '%')
print('rmse 提升:', lgb_rmse_enhance1, '%')

(34340, 37)
lgb_model_mae1: 8.570603826465518
native_mae: 10.578265705206551
lgb_model_rmse1: 20.80648641232052
native_rmse: 26.220882300682653
mae 提升: 18.98 %
rmse 提升: 20.65 %


In [204]:
lgb_pre = model1.predict(X_test)

### test

In [205]:
X = pd.read_csv('./train_data/dam_features.csv')
X_test = X[X.timestamp >= '2025-01-01 00:00:00']
X_test = X_test.dropna()

In [206]:
y_pred = pd.Series(y_pred)
y_test.reset_index(drop=True, inplace=True)
dt = X_test[['timestamp', 'hour']]
dt.reset_index(drop=True, inplace=True)
result = pd.concat([dt, y_test, y_pred], axis=1)
result.columns = ['dt', 'hour', 'target', 'pred']

In [207]:
result.to_csv("LGBM_DAM_result.csv", index=None)

In [208]:
result

,dt,hour,target,pred
0,2025-01-01 06:00:00,1,20.14,19.753001
1,2025-01-01 06:00:00,2,17.90,19.074642
2,2025-01-01 06:00:00,3,16.69,18.640579
3,2025-01-01 06:00:00,4,17.02,17.960322
4,2025-01-01 06:00:00,5,18.21,19.093189
...,...,...,...,...
8177,2025-12-09 06:00:00,20,60.05,37.727234
8178,2025-12-09 06:00:00,21,63.34,36.967250
8179,2025-12-09 06:00:00,22,57.15,34.352606
8180,2025-12-09 06:00:00,23,49.72,30.498739


## CatBoost

In [169]:
def short_term_forecast_CAT(hour=None):
    # 模型配置（当前最佳）
    if hour:
        cat_features = [0, 1, 2, 3, 5, 6, 7, 8, 31, 32]
    else:
        cat_features = [0, 1, 2, 3, 4, 6, 7, 8, 9, 32, 33]
    model = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.2,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        min_data_in_leaf=20,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        loss_function='MAE',
        task_type='GPU',
        random_seed=42,
        thread_count=-1,
        verbose=0,
        allow_writing_files=False,
        # cat_features=cat_features,
    )
    

    X = pd.read_csv('./train_data/dam_features.csv')
    if hour:
        X = X[X.hour==hour]
        _ = X.pop('hour')
    X = X[X.timestamp >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.timestamp < '2025-01-01 00:00:00']
    X_test = X[X.timestamp >= '2025-01-01 00:00:00']
    _ = X_train.pop('timestamp')
    dt = X_test.pop('timestamp')
    y_train = X_train.pop('target')
    y_test = X_test.pop('target')

    # for i in cat_features:
    #     X_train.iloc[:, i] = X_train.iloc[:, i].astype("category")
    #     X_test.iloc[:, i] = X_test.iloc[:, i].astype("category")

    model.fit(
        X_train, y_train
    )
    
    return  model, X_test, y_test, dt

### 01:00 - 24:00

In [170]:
preds = []
y_tests = []
for h in tqdm(range(1, 25)):
    print("************************")
    print(f'开始第{h}小时预测')
    print("************************")
    model1, X_test, y_test, dt = short_term_forecast_CAT(hour=h)

    y_pred = model1.predict(X_test)
    cat_model_mae1 = mean_absolute_error(y_test, y_pred)
    cat_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
    native_mae = mean_absolute_error(y_test, X_test['dam_lag_24h'])
    native_rmse = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
    print("lgb_model_mae1:", cat_model_mae1)
    print("native_mae:", native_mae)

    print("lgb_model_rmse1:", cat_model_rmse1)
    print("native_rmse:", native_rmse)

    cat_mae_enhance1 = round((native_mae - cat_model_mae1) / native_mae * 100, 2)
    cat_rmse_enhance1 = round((native_rmse - cat_model_rmse1) / native_rmse * 100, 2)
    print('mae 提升:', cat_mae_enhance1, '%')
    print('rmse 提升:', cat_rmse_enhance1, '%')
    preds.extend(list(y_pred))
    y_tests.extend(list(y_test))
    print()

  0%|          | 0/24 [00:00<?, ?it/s]

************************
开始第1小时预测
************************
(1431, 36)


Default metric period is 5 because MAE is/are not implemented for GPU


lgb_model_mae1: 5.05104557150313
native_mae: 7.0146627565982405
lgb_model_rmse1: 7.6588235447636315
native_rmse: 10.530246395481928
mae 提升: 27.99 %
rmse 提升: 27.27 %

************************
开始第2小时预测
************************
(1431, 36)


Default metric period is 5 because MAE is/are not implemented for GPU


...

lgb_model_mae1: 14.270870654935054
native_mae: 17.374780058651023
lgb_model_rmse1: 29.1681949041718
native_rmse: 33.8725738069162
mae 提升: 17.86 %
rmse 提升: 13.89 %

************************
开始第23小时预测
************************
(1431, 36)


Default metric period is 5 because MAE is/are not implemented for GPU


lgb_model_mae1: 8.79874861081255
native_mae: 10.374721407624632
lgb_model_rmse1: 13.683615311752986
native_rmse: 15.52517869807102
mae 提升: 15.19 %
rmse 提升: 11.86 %

************************
开始第24小时预测
************************
(1431, 36)


Default metric period is 5 because MAE is/are not implemented for GPU


lgb_model_mae1: 6.366542133017135
native_mae: 7.123958944281523
lgb_model_rmse1: 9.371555247149464
native_rmse: 10.39863975708621
mae 提升: 10.63 %
rmse 提升: 9.88 %



In [171]:
y_pred = preds
y_test = y_tests
xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
print("lgb_model_mae1:", xgb_model_mae1)

print("lgb_model_rmse1:", xgb_model_rmse1)

lgb_model_mae1: 8.616168147749963
lgb_model_rmse1: 21.28036327731396


In [172]:
model1, X_test, y_test, dt = short_term_forecast_CAT()

y_pred = model1.predict(X_test)
lgb_model_mae1 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test['dam_lag_24h'])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test['dam_lag_24h']))
print("lgb_model_mae1:", lgb_model_mae1)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse1)
print("native_rmse:", native_rmse)

lgb_mae_enhance1 = round((native_mae - lgb_model_mae1) / native_mae * 100, 2)
lgb_rmse_enhance1 = round((native_rmse - lgb_model_rmse1) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance1, '%')
print('rmse 提升:', lgb_rmse_enhance1, '%')

(34340, 37)


Default metric period is 5 because MAE is/are not implemented for GPU


lgb_model_mae1: 8.575662316495931
native_mae: 10.578265705206551
lgb_model_rmse1: 21.223247015516
native_rmse: 26.220882300682653
mae 提升: 18.93 %
rmse 提升: 19.06 %


In [173]:
cat_pre = model1.predict(X_test)

## 最终结果

In [200]:
pre = xgb_pre + lgb_pre + cat_pre
pre = pre / 3

In [201]:
mean_absolute_error(y_test, pre)

8.54020795335717

In [202]:
np.sqrt(mean_squared_error(y_test, pre))

20.90615997789408

In [ ]:
xgb_mae = 8.61
xgb_rmse = 20.85

lgb_mae = 8.57
lgb_rmse = 20.81

cat_mae = 8.58
cat_rmse = 21.22

native_mae = 10.58
native_rmse = 26.22